# Calibration and Reliability of Classifier Confidence

**Module 06 · AI Safety (ISO/PAS 8800)** · Notebook 4 of 8

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/06_AI_Safety_ISO_PAS_8800/04_calibration.ipynb)

| Level | Prerequisites | Standards |
|---|---|---|
| Intermediate | [02 Uncertainty types](02_uncertainty_types.ipynb), [03 MC dropout and ensembles](03_mc_dropout_and_deep_ensembles.ipynb) | ISO/PAS 8800:2024, ISO 21448:2022 |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

- Define calibration precisely (top-label confidence vs observed accuracy).
- Measure it with a **reliability diagram**, **expected calibration error (ECE)** and the proper scoring rules **NLL** and **Brier score**, each with a **bootstrap confidence interval**.
- Fit **temperature scaling** correctly (minimise validation NLL over $T>0$), plus Platt scaling and isotonic regression, using a strict train / validation / test protocol.
- Explain what calibration does and does not change (accuracy, ranking, thresholded decisions).
- Use calibrated probabilities for **expected-cost decisions** and see how calibration degrades under **distribution shift**.
- Measure and recalibrate the confidence of a **real object detector on KITTI images**: state the matching rule, use a frame-level bootstrap, analyse slices, and relate the result to the AEB safety goals.

## Introduction

A model predicts "pedestrian" with confidence 0.9. What does that mean?
- **Calibrated:** among all predictions made with confidence 0.9, about 90 % are correct.
- **Miscalibrated:** the true rate may be 60 % or 99 %.

Guo et al. (2017) showed that modern neural networks are often **overconfident**: high accuracy, but confidences above the observed accuracy. Overfitting the negative log-likelihood is one mechanism.

**Why calibration matters for AVs:**
- Downstream components (tracking, sensor fusion, planning) combine probabilities and need them to mean what they say.
- Expected-cost decisions ("brake if the expected cost of not braking exceeds that of braking") are only correct with calibrated probabilities.
- ISO 21448 and ISO/PAS 8800 require the limitations of the perception function to be understood; a confidence score is only useful as evidence if its calibration has been measured on representative data.

In [ ]:
# Course toolkit (tables, formulas and real-data loaders). Installs automatically on Colab.
try:
    import avsafety
except ImportError:
    %pip install -q "avsafety[data] @ git+https://github.com/milinpatel07/Autonomous-Driving_AI-Safety-and-Security.git"
    import avsafety
from avsafety import data as D

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import optimize
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression

rng = np.random.default_rng(42)
torch.manual_seed(42)
torch.set_num_threads(1)              # tiny model: one thread is fastest and reproducible
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
OI = ["#E69F00", "#56B4E9", "#009E73", "#F0E442", "#0072B2", "#D55E00", "#CC79A7", "#000000"]
pd.set_option("display.width", 160)

## 1. Definitions and metrics

**Perfect (top-label) calibration:** $P(\hat Y = Y \mid \hat P = p) = p$ for all $p$, where $\hat Y$ is the predicted class and $\hat P$ its confidence (the maximum class probability).

**Reliability diagram:** bin the predictions by confidence, then plot the mean confidence of each bin (x) against its accuracy (y). The diagonal is perfect calibration; points *below* it are **overconfident**, points *above* it **underconfident**. For two classes the top-label confidence lies in $[0.5, 1]$, so the bins below 0.5 stay empty.

**Expected calibration error** (equal-width bins $B_1,\dots,B_K$ over the confidence):
$$\mathrm{ECE} = \sum_{k=1}^{K} \frac{|B_k|}{n}\,\big|\mathrm{acc}(B_k) - \mathrm{conf}(B_k)\big| .$$
ECE = 0.10 means that the bin-weighted average gap between accuracy and confidence is 10 percentage points. ECE depends on the number and type of bins, is biased upwards for small samples and has high variance, so it should be reported with a confidence interval (Nixon et al., 2019; Kumar et al., 2019). It can also be small for a useless model that always predicts the base rate.

**Proper scoring rules** complement ECE because they reward both calibration and sharpness:
- Negative log-likelihood $\mathrm{NLL} = -\frac1n\sum_i \log \hat p_{i,y_i}$;
- Brier score $= \frac1n\sum_i\sum_c (\hat p_{i,c} - \mathbb 1[y_i=c])^2$ (multi-class form; for two classes it is twice the common binary form $\frac1n\sum_i(\hat p_i - y_i)^2$).

The functions below work on probability matrices of shape (n, classes) and give the same results as `avsafety.uq` in this repository (see repo `avsafety/`).

In [ ]:
def softmax(logits, temperature=1.0):
    z = np.asarray(logits, dtype=np.float64) / temperature
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def reliability_bins(probs, labels, n_bins=15):
    """Per bin: (mean confidence, accuracy, count); top-label confidence, equal-width bins on [0, 1]."""
    conf = probs.max(axis=1)
    correct = (probs.argmax(axis=1) == labels).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(conf, edges[1:-1], right=True), 0, n_bins - 1)
    return [(conf[idx == b].mean() if (idx == b).any() else np.nan,
             correct[idx == b].mean() if (idx == b).any() else np.nan, int((idx == b).sum())) for b in range(n_bins)]

def expected_calibration_error(probs, labels, n_bins=15):
    n = len(labels)
    return float(sum(c / n * abs(a - m) for m, a, c in reliability_bins(probs, labels, n_bins) if c))

def negative_log_likelihood(probs, labels, eps=1e-12):
    return float(-np.mean(np.log(np.clip(probs[np.arange(len(labels)), labels], eps, 1.0))))

def brier_score(probs, labels):
    onehot = np.eye(probs.shape[1])[labels]
    return float(np.mean(np.sum((probs - onehot) ** 2, axis=1)))

def to_two_class(p_pos):
    """Binary P(y=1) -> (n, 2) probability matrix."""
    p_pos = np.asarray(p_pos, dtype=np.float64)
    return np.stack([1 - p_pos, p_pos], axis=1)

# Sanity check on a synthetic, perfectly calibrated source: ECE should be small, and shrink with n
for n in (500, 50_000):
    p = rng.uniform(0.5, 1.0, n)                       # confidences of the predicted class (class 1)
    y = (rng.random(n) < p).astype(int)                # correct with exactly that probability
    print(f"perfectly calibrated source, n = {n:6d}: ECE = {expected_calibration_error(to_two_class(p), y):.4f}")

In [ ]:
def reliability_diagram(probs, labels, title, n_bins=15, ax=None, ax_hist=None):
    stats = reliability_bins(probs, labels, n_bins)
    width = 1.0 / n_bins
    centres = (np.arange(n_bins) + 0.5) * width
    ece = expected_calibration_error(probs, labels, n_bins)
    ax.plot([0, 1], [0, 1], "k--", lw=1.5, label="Perfect calibration")
    for k, (m, acc, cnt) in enumerate(stats):
        if not cnt:
            continue
        ax.bar(centres[k], acc, width=width * 0.95, color=OI[4], edgecolor="black", alpha=0.8,
               label="Accuracy in bin" if k == np.nanargmax([s[2] for s in stats]) else None)
        gap_lo, gap_hi = sorted([acc, m])
        ax.bar(centres[k], gap_hi - gap_lo, bottom=gap_lo, width=width * 0.95,
               color=OI[5] if m > acc else OI[2], alpha=0.45, edgecolor="black", hatch="//" if m > acc else "..")
    ax.bar([np.nan], [np.nan], color=OI[5], alpha=0.45, hatch="//", label="Gap: overconfident (conf > acc)")
    ax.bar([np.nan], [np.nan], color=OI[2], alpha=0.45, hatch="..", label="Gap: underconfident (conf < acc)")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("Top-label confidence (mean in bin)"); ax.set_ylabel("Accuracy in bin (fraction correct)")
    ax.set_title(f"{title}\nECE = {ece:.3f} ({n_bins} bins)", fontweight="bold"); ax.legend(fontsize=8, loc="upper left")
    if ax_hist is not None:
        ax_hist.bar(centres, [s[2] for s in stats], width=width * 0.95, color=OI[1], edgecolor="black")
        ax_hist.set_xlim(0, 1); ax_hist.set_xlabel("Top-label confidence"); ax_hist.set_ylabel("Number of predictions")
        ax_hist.set_title("Confidence histogram")
    return ece

## 2. A model that becomes overconfident

Synthetic binary task with 5 % label noise (`flip_y`). Data are split **once** into training (50 %), **validation** (25 %, used only to fit calibrators) and **test** (25 %, used only for reporting). The network is trained long enough to overfit the NLL on the training set, the mechanism behind overconfidence described by Guo et al. (2017).

> **Illustrative data:** values below are synthetic, chosen for teaching; they are not measurements.

In [ ]:
X, y = make_classification(n_samples=4000, n_features=20, n_informative=10, n_redundant=5,
                           flip_y=0.05, class_sep=1.0, random_state=42)
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.5, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp)
print(f"Train {len(X_train)}, validation {len(X_val)}, test {len(X_test)}")

class BinaryClassifier(nn.Module):
    def __init__(self, d=20, h=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)                   # logit of P(y = 1)

tt = lambda a: torch.as_tensor(a, dtype=torch.float32)
model = BinaryClassifier()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)      # no weight decay, no early stopping: overfits NLL
g = torch.Generator().manual_seed(0)
hist = {"train": [], "val": []}
Xtr_t, ytr_t, Xv_t, yv_t = tt(X_train), tt(y_train), tt(X_val), tt(y_val)
for epoch in range(60):
    model.train()
    perm = torch.randperm(len(Xtr_t), generator=g)
    for i in range(0, len(perm), 64):
        idx = perm[i:i + 64]
        opt.zero_grad(); loss = F.binary_cross_entropy_with_logits(model(Xtr_t[idx]), ytr_t[idx]); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        hist["train"].append(F.binary_cross_entropy_with_logits(model(Xtr_t), ytr_t).item())
        hist["val"].append(F.binary_cross_entropy_with_logits(model(Xv_t), yv_t).item())

with torch.no_grad():
    z_val = model(Xv_t).double().numpy()
    z_test = model(tt(X_test)).double().numpy()
logits2 = lambda z: np.stack([np.zeros_like(z), z], axis=1)     # binary logit -> 2-class logits [0, z]

plt.figure(figsize=(9, 4))
plt.plot(hist["train"], label="Training NLL"); plt.plot(hist["val"], label="Validation NLL")
plt.xlabel("Epoch"); plt.ylabel("Mean NLL (nats)"); plt.title("Training NLL keeps falling, validation NLL rises: overfitting")
plt.legend(); plt.tight_layout(); plt.show()
best = int(np.argmin(hist["val"]))
print(f"Validation NLL minimum at epoch {best + 1} ({hist['val'][best]:.3f}); final {hist['val'][-1]:.3f}")

In [ ]:
probs_raw = softmax(logits2(z_test))
acc_raw = np.mean(probs_raw.argmax(1) == y_test)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ece_raw = reliability_diagram(probs_raw, y_test, "Uncalibrated network (test set)", ax=ax1, ax_hist=ax2)
plt.tight_layout(); plt.show()
mean_conf = probs_raw.max(1).mean()
print(f"Test accuracy {acc_raw:.3f}, mean confidence {mean_conf:.3f} -> "
      f"{'overconfident' if mean_conf > acc_raw else 'underconfident'} by {abs(mean_conf - acc_raw):.3f}")
print(f"ECE {ece_raw:.4f}, NLL {negative_log_likelihood(probs_raw, y_test):.4f}, Brier {brier_score(probs_raw, y_test):.4f}")

## 3. Post-hoc calibration methods

All calibrators are **fitted on the validation set** and **evaluated on the test set**.

### 3.1 Temperature scaling (Guo et al., 2017)

Divide the logits by a single scalar $T>0$: $\hat p = \mathrm{softmax}(z/T)$.
- $T>1$ softens the probabilities (corrects overconfidence), $T<1$ sharpens them (corrects underconfidence).
- $T$ is fitted by **minimising the validation NLL**. ECE is not used as the objective: it is binned, non-smooth and biased.
- Optimise over $\log T$, so that $T$ stays positive, and in float64. (A common bug is to optimise a float32 loss with finite-difference gradients: the step of $10^{-8}$ in $T$ does not change the float32 loss, the gradient is zero and the optimiser stops at $T=1$.)
- The argmax, and therefore accuracy at the 0.5 threshold, is unchanged; decisions at any other threshold do change.
- A strong in-distribution baseline; its calibration degrades under distribution shift (Ovadia et al., 2019).

In [ ]:
def fit_temperature(val_logits, val_labels):
    """T > 0 minimising validation NLL, optimised over log T in float64 (same as avsafety.uq.fit_temperature)."""
    nll = lambda log_t: negative_log_likelihood(softmax(val_logits, np.exp(log_t)), val_labels)
    res = optimize.minimize_scalar(nll, bounds=(-3.0, 3.0), method="bounded")
    return float(np.exp(res.x))

T_opt = fit_temperature(logits2(z_val), y_val)
probs_ts = softmax(logits2(z_test), T_opt)

Ts = np.exp(np.linspace(np.log(0.3), np.log(10), 80))
val_nll = [negative_log_likelihood(softmax(logits2(z_val), T), y_val) for T in Ts]
fig, (ax0, ax1, ax2) = plt.subplots(1, 3, figsize=(19, 5))
ax0.plot(Ts, val_nll, color=OI[4], lw=2)
ax0.axvline(T_opt, color=OI[5], ls="--", label=f"T* = {T_opt:.2f}")
ax0.axvline(1.0, color="grey", ls=":", label="T = 1 (uncalibrated)")
ax0.set_xscale("log"); ax0.set_xlabel("Temperature T (log scale)"); ax0.set_ylabel("Validation NLL (nats)")
ax0.set_title("Temperature scaling objective"); ax0.legend()
ece_ts = reliability_diagram(probs_ts, y_test, f"Temperature scaled, T = {T_opt:.2f} (test set)", ax=ax1, ax_hist=ax2)
plt.tight_layout(); plt.show()
print(f"Fitted T = {T_opt:.3f} ({'model was overconfident' if T_opt > 1 else 'model was underconfident'}).")
print(f"Accuracy unchanged: {np.mean(probs_ts.argmax(1) == y_test):.3f} vs {acc_raw:.3f}")
print(f"Test ECE {ece_raw:.4f} -> {ece_ts:.4f}; NLL {negative_log_likelihood(probs_raw, y_test):.4f} -> "
      f"{negative_log_likelihood(probs_ts, y_test):.4f}")

### 3.2 Platt scaling

Fit $\hat p = \sigma(a z + b)$ on the validation logits by unregularised maximum likelihood. Temperature scaling is the special case $a = 1/T,\ b = 0$. The extra bias $b$ shifts the 0.5 decision boundary, so **accuracy can change**. (Platt's original 1999 method also used smoothed targets; plain maximum likelihood is shown here. scikit-learn's `LogisticRegression` applies L2 regularisation by default, which shrinks $a$; we therefore fit directly.)

### 3.3 Isotonic regression

A non-parametric, monotone, piecewise-constant map from score to probability (Zadrozny & Elkan, 2002). Very flexible, but it needs more validation data, can overfit, and maps ranges of scores to ties, which can change accuracy and ranking (AUROC).

In [ ]:
def fit_platt(z, y):
    def nll(params):
        a, b = params
        s = a * z + b
        return np.mean(np.logaddexp(0, s) - y * s)          # stable binary NLL
    res = optimize.minimize(nll, x0=[1.0, 0.0], method="BFGS")
    return res.x

a_pl, b_pl = fit_platt(z_val, y_val)
probs_platt = to_two_class(1 / (1 + np.exp(-(a_pl * z_test + b_pl))))

iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0).fit(1 / (1 + np.exp(-z_val)), y_val)
probs_iso = to_two_class(iso.predict(1 / (1 + np.exp(-z_test))))

print(f"Platt: a = {a_pl:.3f} (equivalent T = {1 / a_pl:.2f}), b = {b_pl:.3f}")
print(f"Isotonic: {len(iso.X_thresholds_)} breakpoints; distinct output values on test: {len(np.unique(probs_iso[:, 1]))}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
reliability_diagram(probs_platt, y_test, "Platt scaling (test set)", ax=axes[0])
reliability_diagram(probs_iso, y_test, "Isotonic regression (test set)", ax=axes[1])
plt.tight_layout(); plt.show()

## 4. Comparison with bootstrap confidence intervals

The test set has 1,000 samples. To see whether differences are larger than the sampling noise, we compute **95 % percentile bootstrap intervals** (2,000 resamples of the test set) and, for the comparison with the uncalibrated model, a **paired** bootstrap of the difference.

In [ ]:
methods = {"Uncalibrated": probs_raw, "Temperature scaling": probs_ts,
           "Platt scaling": probs_platt, "Isotonic regression": probs_iso}
metrics = {"ECE": expected_calibration_error, "NLL": negative_log_likelihood, "Brier": brier_score,
           "Accuracy": lambda p, y: float(np.mean(p.argmax(1) == y))}

B = 2000
boot_idx = rng.integers(0, len(y_test), size=(B, len(y_test)))
boot = {m: {k: np.array([f(P[i], y_test[i]) for i in boot_idx]) for k, f in metrics.items()} for m, P in methods.items()}

rows = []
for m, P in methods.items():
    row = {"method": m}
    for k, f in metrics.items():
        lo, hi = np.percentile(boot[m][k], [2.5, 97.5])
        row[k] = f"{f(P, y_test):.4f} [{lo:.4f}, {hi:.4f}]"
    if m != "Uncalibrated":
        for k in ("ECE", "NLL"):
            d = boot[m][k] - boot["Uncalibrated"][k]
            lo, hi = np.percentile(d, [2.5, 97.5])
            row[f"Δ{k} vs uncal. (95 % CI)"] = f"{d.mean():+.4f} [{lo:+.4f}, {hi:+.4f}]" + (" *" if hi < 0 or lo > 0 else "")
    rows.append(row)
table = pd.DataFrame(rows).set_index("method").fillna("")
print("Test-set metrics, point estimate [95 % bootstrap CI]; * = paired CI of the difference excludes 0")
print(table.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
names = list(methods)
for ax, k in zip(axes, ["ECE", "NLL", "Brier"]):
    est = np.array([metrics[k](methods[m], y_test) for m in names])
    ci = np.array([np.percentile(boot[m][k], [2.5, 97.5]) for m in names])
    order = np.argsort(est)                                       # colour by rank, best = darkest
    colours = [None] * len(names)
    for rank, i in enumerate(order):
        colours[i] = [OI[2], OI[1], OI[0], OI[5]][rank]
    ax.bar(range(len(names)), est, yerr=[est - ci[:, 0], ci[:, 1] - est], capsize=5, color=colours, edgecolor="black")
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=25, ha="right")
    ax.set_ylabel(k); ax.set_title(f"{k} on test set (95 % bootstrap CI)\nbest: {names[order[0]]}")
plt.tight_layout(); plt.show()

ranking = sorted(names, key=lambda m: metrics["NLL"](methods[m], y_test))
print("Ranking by test NLL (proper scoring rule):", " < ".join(ranking))
ranking_ece = sorted(names, key=lambda m: metrics["ECE"](methods[m], y_test))
print("Ranking by test ECE:                        ", " < ".join(ranking_ece))
if ranking[0] != ranking_ece[0]:
    print("The two rankings disagree on the best method, a reminder that ECE alone is not a sufficient criterion.")

## 5. Calibration and safety decisions

### 5.1 Thresholds vs expected cost
Two different uses of a score must not be confused:

1. **A threshold meeting a target miss rate** (false-negative rate, FNR) is chosen on validation data, e.g. from the ROC curve or with conformal risk control ([05](05_conformal_prediction.ipynb)). This does not need calibrated probabilities, because any monotone score works.
2. **Expected-cost decisions** do need calibration. If missing a pedestrian costs $C_{FN}$ and an unnecessary emergency brake costs $C_{FP}$ (rear-end risk, loss of trust), braking is optimal when
$$P(\text{ped})\,C_{FN} > \big(1 - P(\text{ped})\big)\,C_{FP} \iff P(\text{ped}) > t^* = \frac{C_{FP}}{C_{FP} + C_{FN}} .$$
This rule is only optimal if $P(\text{ped})$ is calibrated.

The costs below are **illustrative** (a missed pedestrian weighted 20 times an unnecessary brake), and class 1 plays the role of "pedestrian".

In [ ]:
C_FN, C_FP = 20.0, 1.0
t_star = C_FP / (C_FP + C_FN)

def confusion(p_pos, y, thr):
    d = p_pos > thr
    tp = np.sum(d & (y == 1)); fp = np.sum(d & (y == 0)); fn = np.sum(~d & (y == 1)); tn = np.sum(~d & (y == 0))
    return tp, fp, fn, tn

fig, axes = plt.subplots(1, 4, figsize=(19, 4.3))
rows = []
for ax, (m, P) in zip(axes, methods.items()):
    tp, fp, fn, tn = confusion(P[:, 1], y_test, t_star)
    cost = (C_FN * fn + C_FP * fp) / len(y_test)
    rows.append((m, fn / (tp + fn), fp / (fp + tn), cost))
    cm = np.array([[tn, fp], [fn, tp]])
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=16, fontweight="bold",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred. negative", "Pred. positive"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["True negative", "True positive"])
    ax.grid(False)
    ax.set_title(f"{m}\nFNR {fn / (tp + fn):.3f}, FPR {fp / (fp + tn):.3f}, cost {cost:.3f}", fontsize=10)
plt.suptitle(f"Decisions at the expected-cost threshold t* = {t_star:.3f} (C_FN = {C_FN:g}, C_FP = {C_FP:g})", fontweight="bold")
plt.tight_layout(); plt.show()

# Best achievable cost with the threshold tuned on the TEST set (oracle, for reference only)
oracle = min((C_FN * confusion(probs_raw[:, 1], y_test, th)[2] + C_FP * confusion(probs_raw[:, 1], y_test, th)[1]) / len(y_test)
             for th in np.unique(probs_raw[:, 1]))
print(f"{'method':22s} {'FNR':>6s} {'FPR':>6s} {'mean cost':>10s}")
for m, fnr, fpr, cost in rows:
    print(f"{m:22s} {fnr:6.3f} {fpr:6.3f} {cost:10.4f}")
best = min(rows, key=lambda r: r[3])
print(f"\nLowest cost at t*: {best[0]} ({best[3]:.4f}); oracle threshold on test data: {oracle:.4f}.")
print("With miscalibrated probabilities, the theoretically optimal threshold t* is applied to the wrong scale.")

### 5.2 Calibration under distribution shift

Calibration is measured on a particular distribution. Below, the test inputs are perturbed with increasing Gaussian noise, a stand-in for sensor degradation (rain, dirt). The temperature fitted on clean validation data is kept fixed.

In [ ]:
shift_levels = [0.0, 0.5, 1.0, 1.5, 2.0]
shift_rows = []
for s in shift_levels:
    X_s = X_test + s * rng.standard_normal(X_test.shape)
    with torch.no_grad():
        z_s = model(tt(X_s)).double().numpy()
    for name, T in [("Uncalibrated", 1.0), ("Temperature scaling", T_opt)]:
        P = softmax(logits2(z_s), T)
        shift_rows.append((s, name, np.mean(P.argmax(1) == y_test), P.max(1).mean(), expected_calibration_error(P, y_test)))
shift = pd.DataFrame(shift_rows, columns=["noise std", "method", "accuracy", "mean confidence", "ECE"])

fig, ax = plt.subplots(figsize=(9, 4.8))
for name, c in [("Uncalibrated", OI[5]), ("Temperature scaling", OI[4])]:
    d = shift[shift.method == name]
    ax.plot(d["noise std"], d["ECE"], "o-", color=c, lw=2, label=f"{name}: ECE")
d = shift[shift.method == "Temperature scaling"]
ax.plot(d["noise std"], 1 - d["accuracy"], "s:", color="grey", label="Error rate (same for both)")
ax.set_xlabel("Std of added input noise (feature units)"); ax.set_ylabel("ECE / error rate")
ax.set_title("Calibration fitted on clean data, evaluated under shift"); ax.legend()
plt.tight_layout(); plt.show()
print(shift.round(3).to_string(index=False))
ts = shift[shift.method == "Temperature scaling"].set_index("noise std")
print(f"\nWith T fixed, ECE grows from {ts.loc[0.0, 'ECE']:.3f} (no shift) to {ts.loc[shift_levels[-1], 'ECE']:.3f} "
      f"at noise std {shift_levels[-1]}; mean confidence {ts.loc[shift_levels[-1], 'mean confidence']:.3f} vs accuracy "
      f"{ts.loc[shift_levels[-1], 'accuracy']:.3f}.")

Calibration is a property of a model *and a distribution*. For a safety argument this means: report calibration per ODD slice (day/night, weather, sensor condition), monitor it in the field with labelled audits, and do not assume that a calibrator fitted on clean development data transfers to degraded conditions (Ovadia et al., 2019).

## 6. Real data: calibration of a detector's confidence on KITTI

Sections 2–5 used a synthetic model so that every quantity was controlled. We now apply the same tools to a real perception component: torchvision's **Faster R-CNN with a MobileNetV3-Large FPN backbone** (Ren et al., 2017; Howard et al., 2019), trained on COCO (Lin et al., 2014) and applied **without fine-tuning** to 40 KITTI camera frames. For every box the detector reports a score in $[0, 1]$. The question an AEB developer has to answer is: *when the detector reports 0.9 for a pedestrian, is there a pedestrian at that location 90 % of the time?*

**What "correct" means.** For a detector, calibration is defined relative to a matching rule (Küppers et al., 2020). We use the KITTI rule: a detection is a **true positive (TP)** if it overlaps a not yet matched ground-truth object of its class with IoU ≥ 0.7 (car) or ≥ 0.5 (pedestrian). Detections are **ignored** if they fall on objects that KITTI does not evaluate for that class (DontCare regions; vans for the car class; cyclists and sitting persons for the pedestrian class; ground truth below 25 px, heavily occluded or truncated) or if they are themselves smaller than 25 px. Everything else is a **false positive (FP)**. The calibration target is therefore $P(\text{TP}\mid s)$, the precision at score $s$, and the reliability diagram plots the score itself (not the top-label confidence of Sections 1–5) against the observed TP rate. The corresponding ECE is the detection ECE of Küppers et al. (2020) without position dependence.

### Data and licence

KITTI object-detection training set (Geiger et al., 2012), images, 2D labels and calibration fetched on demand by `avsafety.data` and cached; licence CC BY-NC-SA 3.0 (non-commercial use). `FRAMES_PED` are 20 frames with at least three evaluable pedestrians and `FRAMES_CAR` 20 frames with at least four evaluable cars and no pedestrians, drawn once with seed 8800 such that no two frame indices are closer than six (consecutive KITTI indices can come from the same drive). The detector weights (about 75 MB) are downloaded once by torchvision.

In [ ]:
import os
import time
import torchvision
from scipy import stats

print(D.attribution("KITTI"))
FRAMES_PED = [336, 1068, 1169, 1428, 1496, 1931, 2496, 2920, 4665, 4950,
              4963, 5059, 5571, 5666, 6412, 6475, 6578, 6749, 7286, 7412]
FRAMES_CAR = [361, 389, 819, 850, 1270, 1591, 1646, 2057, 2153, 2349,
              2444, 2806, 2858, 3016, 3781, 4447, 5831, 6532, 7263, 7475]
COCO_TO_KITTI = {1: "Pedestrian", 3: "Car"}                     # COCO person -> Pedestrian, COCO car -> Car
IOU_MIN = {"Car": 0.7, "Pedestrian": 0.5}                      # KITTI evaluation thresholds
NEIGHBOUR = {"Car": {"Van"}, "Pedestrian": {"Person_sitting", "Cyclist"}}   # ignored, neither TP nor FP
HEIGHT_M = {"Car": 1.5, "Pedestrian": 1.75}                    # typical heights for a pinhole distance proxy

torch.set_num_threads(min(4, os.cpu_count()))                   # the detector benefits from several threads
detector = torchvision.models.detection.fasterrcnn_mobilenet_v3_large_fpn(weights="DEFAULT").eval()

def run_detector(image):
    x = torch.from_numpy(image.copy()).permute(2, 0, 1).float() / 255
    with torch.no_grad():
        out = detector([x])[0]
    return {k: v.numpy() for k, v in out.items()}

def box_iou(a, b):
    """IoU matrix (len(a) x len(b)) and intersection over the area of a, for [x1, y1, x2, y2] boxes."""
    a, b = np.asarray(a, float).reshape(-1, 4), np.asarray(b, float).reshape(-1, 4)
    iw = np.clip(np.minimum(a[:, None, 2], b[None, :, 2]) - np.maximum(a[:, None, 0], b[None, :, 0]), 0, None)
    ih = np.clip(np.minimum(a[:, None, 3], b[None, :, 3]) - np.maximum(a[:, None, 1], b[None, :, 1]), 0, None)
    inter = iw * ih
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9), inter / (area_a[:, None] + 1e-9)

def match_frame(fid, frame, out, iou_min=IOU_MIN):
    """KITTI-style greedy matching per class, highest score first. Returns detection rows and GT rows."""
    det_rows, gt_rows = [], []
    focal = frame.calib["P2"][0, 0]
    as_box = lambda labels: [[l["x1"], l["y1"], l["x2"], l["y2"]] for l in labels]
    for coco_id, cls in COCO_TO_KITTI.items():
        keep = out["labels"] == coco_id
        order = np.argsort(-out["scores"][keep])
        boxes, scores = out["boxes"][keep][order], out["scores"][keep][order]
        care = [l for l in frame.labels if l["type"] == cls and D.kitti_difficulty(l) != "Ignored"]
        ignore = [l for l in frame.labels if (l["type"] == cls and D.kitti_difficulty(l) == "Ignored")
                  or l["type"] in NEIGHBOUR[cls]]
        dontcare = [l for l in frame.labels if l["type"] == "DontCare"]
        iou_c, _ = box_iou(boxes, as_box(care))
        iou_i, _ = box_iou(boxes, as_box(ignore))
        _, cover_dc = box_iou(boxes, as_box(dontcare))
        matched_score = np.zeros(len(care))
        taken = np.zeros(len(care), bool)
        for j, (box, s) in enumerate(zip(boxes, scores)):
            h = box[3] - box[1]
            cand = np.where(~taken & (iou_c[j] >= iou_min[cls]))[0]
            if len(cand):
                g = cand[np.argmax(iou_c[j, cand])]; taken[g] = True; matched_score[g] = s
                status, dist, occ = "TP", float(np.hypot(care[g]["x"], care[g]["z"])), care[g]["occluded"]
            else:
                ignored = ((iou_i.shape[1] and iou_i[j].max() >= iou_min[cls])
                           or (cover_dc.shape[1] and cover_dc[j].max() >= 0.5) or h < 25)
                status, dist, occ = ("ignored" if ignored else "FP"), np.nan, -1
            det_rows.append(dict(frame=fid, cls=cls, score=float(s), status=status, box=box, box_h=h,
                                 best_iou=float(iou_c[j].max()) if iou_c.shape[1] else 0.0,
                                 dist_proxy=focal * HEIGHT_M[cls] / h, dist_gt=dist, occluded=occ))
        for g, l in enumerate(care):
            gt_rows.append(dict(frame=fid, cls=cls, occluded=l["occluded"], dist=float(np.hypot(l["x"], l["z"])),
                                score=matched_score[g]))
    return det_rows, gt_rows

t0 = time.perf_counter()
frames, outputs, det_rows, gt_rows = {}, {}, [], []
for fid in FRAMES_PED + FRAMES_CAR:
    try:
        frames[fid] = D.kitti_object(fid, parts=("image", "labels", "calib"))
    except D.DataUnavailable as exc:
        raise SystemExit(f"KITTI could not be downloaded ({exc}). Run the notebook with internet access.")
    outputs[fid] = run_detector(frames[fid].image)
    d_rows, g_rows = match_frame(fid, frames[fid], outputs[fid])
    det_rows += d_rows; gt_rows += g_rows
det_all = pd.DataFrame(det_rows); gt = pd.DataFrame(gt_rows)
det = det_all[det_all.status != "ignored"].copy()
det["tp"] = (det.status == "TP").astype(int)
print(f"{len(frames)} frames processed in {time.perf_counter() - t0:.0f} s")
print(det_all.groupby(["cls", "status"]).size().unstack().to_string())
print(f"Evaluated detections (TP + FP): {len(det)}; evaluable ground-truth objects: "
      f"{(gt.cls == 'Car').sum()} cars, {(gt.cls == 'Pedestrian').sum()} pedestrians")

In [ ]:
SHOW = [1428, 389]                                   # one pedestrian frame, one car frame
STYLE = {"TP": (OI[4], "-"), "FP": (OI[5], "-"), "ignored": ("grey", ":")}
fig, axes = plt.subplots(2, 1, figsize=(15, 9.5))
for ax, fid in zip(axes, SHOW):
    fr, rows = frames[fid], det_all[(det_all.frame == fid) & (det_all.score >= 0.3)]
    ax.imshow(fr.image)
    for l in fr.labels:
        if l["type"] in ("Car", "Pedestrian") and D.kitti_difficulty(l) != "Ignored":
            ax.add_patch(plt.Rectangle((l["x1"], l["y1"]), l["x2"] - l["x1"], l["y2"] - l["y1"],
                                       fill=False, ec=OI[2], lw=3, alpha=0.9))
    for _, r in rows.iterrows():
        (x1, y1, x2, y2), (col, ls) = r.box, STYLE[r.status]
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=col, lw=1.6, ls=ls))
        ax.text(x1, y1 - 3, f"{r.score:.2f}", color=col, fontsize=8, fontweight="bold",
                bbox=dict(facecolor="white", alpha=0.6, pad=0.5, lw=0))
    ax.set_title(f"KITTI frame {fid:06d}: detections with score >= 0.3 (blue TP, vermillion FP, grey dotted ignored; "
                 f"thick green = evaluable ground truth)", fontsize=10)
    ax.axis("off")
plt.tight_layout(); plt.show()

### 6.1 Reliability of the raw scores

Detections in one image are not independent: they share lighting, the same pedestrian group, the same parked-car row. A bootstrap that resamples individual detections would understate the uncertainty, so all confidence intervals below come from a **cluster bootstrap over frames** (resample whole frames with replacement, 2,000 times). Bins are 10 equal-width score bins.

In [ ]:
def det_bins(p, y, n_bins=10):
    """Per score bin: (mean score, TP rate, count) with equal-width bins on [0, 1]."""
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1], right=True), 0, n_bins - 1)
    return [(p[idx == b].mean(), y[idx == b].mean(), int((idx == b).sum())) if (idx == b).any()
            else (np.nan, np.nan, 0) for b in range(n_bins)]

def det_ece(p, y, n_bins=10):
    """Detection ECE: sum_b |B_b|/n * |TP rate(B_b) - mean score(B_b)|."""
    return float(sum(c / len(y) * abs(a - m) for m, a, c in det_bins(p, y, n_bins) if c))

def frame_bootstrap(df, stat, B=2000, seed=0):
    """Cluster bootstrap: resample frames, not detections. Returns the B bootstrap values of stat(df_b)."""
    r = np.random.default_rng(seed)
    groups = [g.index.values for _, g in df.groupby("frame")]
    picks = r.integers(0, len(groups), size=(B, len(groups)))
    return np.array([stat(df.loc[np.concatenate([groups[i] for i in pick])]) for pick in picks])

def clopper_pearson(k, n, level=0.95):
    """Exact binomial confidence interval for a proportion k/n."""
    a = (1 - level) / 2
    lo = stats.beta.ppf(a, k, n - k + 1) if k > 0 else 0.0
    hi = stats.beta.ppf(1 - a, k + 1, n - k) if k < n else 1.0
    return lo, hi

def det_reliability_plot(ax, p, y, title, colour=OI[4]):
    stats_ = det_bins(p, y)
    m, a, c = map(np.array, zip(*stats_))
    ok = c > 0
    ax.plot([0, 1], [0, 1], "k--", lw=1.2, label="Perfect calibration")
    ax.bar((np.arange(10) + 0.5)[ok] / 10, a[ok], width=0.095, color=colour, edgecolor="black", alpha=0.8,
           label="TP rate in bin")
    ax.plot(m[ok], a[ok], "o", color="black", ms=4, label="(mean score, TP rate)")
    for x_, a_, c_ in zip(m[ok], a[ok], c[ok]):
        ax.annotate(f"{c_}", (x_, a_), textcoords="offset points", xytext=(0, 5), ha="center", fontsize=7)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
    ax.set_xlabel("Detection score (or calibrated probability)"); ax.set_ylabel("Fraction of detections that are TP")
    ax.set_title(f"{title}\nD-ECE = {det_ece(p, y):.3f}, n = {len(y)}", fontsize=10); ax.legend(fontsize=7, loc="upper left")

det = det.reset_index(drop=True)
ece_boot = frame_bootstrap(det, lambda d: det_ece(d.score.values, d.tp.values))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
det_reliability_plot(axes[0], det.score.values, det.tp.values, "All 40 frames, raw scores")
for ax, cls, col in zip(axes[1:], ["Pedestrian", "Car"], [OI[5], OI[0]]):
    d = det[det.cls == cls]
    det_reliability_plot(ax, d.score.values, d.tp.values, f"{cls} detections, raw scores", colour=col)
plt.tight_layout(); plt.show()

lo, hi = np.percentile(ece_boot, [2.5, 97.5])
hi_conf = det[det.score >= 0.9]
k, n = int(hi_conf.tp.sum()), len(hi_conf)
cp = clopper_pearson(k, n)
print(f"D-ECE of raw scores: {det_ece(det.score.values, det.tp.values):.3f} "
      f"[95 % frame-bootstrap CI {lo:.3f}, {hi:.3f}]")
print(f"Detections with score >= 0.9: {n}, of which {k} TP -> precision {k / n:.2f} "
      f"[95 % CI {cp[0]:.2f}, {cp[1]:.2f}] while their mean score is {hi_conf.score.mean():.2f}")
fp_hi = det[(det.score >= 0.9) & (det.tp == 0)]
loc = (fp_hi.best_iou >= 0.3).mean()
print(f"Of the {len(fp_hi)} FPs with score >= 0.9, {loc:.0%} overlap a real object of the same class with "
      f"0.3 <= IoU < threshold (localisation errors or duplicates), {1 - loc:.0%} are on background.")

### 6.2 Recalibration: fit on calibration frames, evaluate on held-out frames

The split is by **frame**, alternating within each list, so that both halves contain pedestrian and car scenes and no image contributes to both. The detector only exposes the score $s$, so the calibrators work on its log-odds $z = \log\frac{s}{1-s}$:

- **Temperature scaling:** $\sigma(z/T)$, one parameter (the same `fit_temperature` as in Section 3.1, applied to the two-class logits $[0, z]$);
- **Platt scaling:** $\sigma(az + b)$, two parameters;
- **Isotonic regression:** monotone, non-parametric; its output is clipped to $[0.001, 0.999]$ so that a single surprise does not produce an infinite NLL.

With a few hundred detections this is a small calibration set; the frame-bootstrap intervals show how much that matters.

In [ ]:
CAL_FRAMES = set(FRAMES_PED[::2] + FRAMES_CAR[::2])
cal, ev = det[det.frame.isin(CAL_FRAMES)].reset_index(drop=True), det[~det.frame.isin(CAL_FRAMES)].reset_index(drop=True)
logit = lambda s: np.log(np.clip(s, 1e-6, 1 - 1e-6) / (1 - np.clip(s, 1e-6, 1 - 1e-6)))
z_cal, z_ev = logit(cal.score.values), logit(ev.score.values)

T_det = fit_temperature(logits2(z_cal), cal.tp.values)
a_det, b_det = fit_platt(z_cal, cal.tp.values)
iso_det = IsotonicRegression(out_of_bounds="clip", y_min=0.001, y_max=0.999).fit(cal.score.values, cal.tp.values)
calibrators = {
    "Raw score": lambda s: s,
    f"Temperature (T = {T_det:.2f})": lambda s: 1 / (1 + np.exp(-logit(s) / T_det)),
    f"Platt (a = {a_det:.2f}, b = {b_det:.2f})": lambda s: 1 / (1 + np.exp(-(a_det * logit(s) + b_det))),
    "Isotonic": lambda s: iso_det.predict(s),
}
print(f"Calibration split: {len(cal)} detections ({cal.tp.sum()} TP) from {len(CAL_FRAMES)} frames; "
      f"held-out split: {len(ev)} detections ({ev.tp.sum()} TP) from {ev.frame.nunique()} frames")

def det_metrics(d, f):
    p, y = f(d.score.values), d.tp.values
    return {"D-ECE": det_ece(p, y), "NLL": negative_log_likelihood(to_two_class(p), y),
            "Brier": brier_score(to_two_class(p), y)}

rows = []
for name, f in calibrators.items():
    point = det_metrics(ev, f)
    boot = frame_bootstrap(ev, lambda d: list(det_metrics(d, f).values()), B=1000, seed=1)
    lo_, hi_ = np.percentile(boot, [2.5, 97.5], axis=0)
    rows.append({"method": name, **{k_: f"{v:.3f} [{l:.3f}, {h:.3f}]" for (k_, v), l, h in zip(point.items(), lo_, hi_)}})
print("Held-out frames, point estimate [95 % frame-bootstrap CI]")
print(pd.DataFrame(rows).set_index("method").to_string())

fig, axes = plt.subplots(1, 4, figsize=(21, 4.8))
for ax, (name, f) in zip(axes, calibrators.items()):
    det_reliability_plot(ax, f(ev.score.values), ev.tp.values, f"{name}, held-out frames")
plt.tight_layout(); plt.show()

# What does a high-confidence claim mean after recalibration?
best = min(list(calibrators)[1:], key=lambda m: negative_log_likelihood(to_two_class(calibrators[m](cal.score.values)),
                                                                         cal.tp.values))
p_ev = calibrators[best](ev.score.values)
for name, p in [("Raw score", ev.score.values), (best, p_ev)]:
    sel = p >= 0.8
    k, n = int(ev.tp.values[sel].sum()), int(sel.sum())
    cp = clopper_pearson(k, n) if n else (np.nan, np.nan)
    print(f"{name:32s}: {n:3d} held-out detections with p >= 0.8, mean p {p[sel].mean():.2f}, "
          f"observed TP rate {k / max(n, 1):.2f} [95 % CI {cp[0]:.2f}, {cp[1]:.2f}]")
print(f"(The calibrator shown second was chosen by NLL on the calibration split, not on held-out data.)")

### 6.3 Calibration per slice, and the definition of "correct"

A calibrator fitted on the pooled data is calibrated *on average over that mixture*. An AEB safety argument needs the slices that matter: pedestrians separately from cars, and near objects separately from far ones. The distance of a false positive is unknown (there is no object), so slices use a pinhole proxy $d \approx f\,H/h$ from the box height $h$ in pixels, the focal length $f$ from the KITTI calibration, and a typical object height $H$ (1.75 m pedestrian, 1.5 m car). Occlusion is defined only for real objects, so it is analysed through the **recall** at an operating threshold instead.

In [ ]:
ev = ev.assign(p_cal=calibrators[best](ev.score.values),
               band=pd.cut(ev.dist_proxy, [0, 15, 30, np.inf], labels=["< 15 m", "15-30 m", "> 30 m"]))
rows = []
for (cls, band), d in ev.groupby(["cls", "band"], observed=True):
    k, n = int(d.tp.sum()), len(d)
    lo_, hi_ = clopper_pearson(k, n)
    rows.append(dict(cls=cls, band=band, n=n, mean_raw=d.score.mean(), mean_cal=d.p_cal.mean(),
                     tp_rate=k / n, ci=f"[{lo_:.2f}, {hi_:.2f}]", gap_cal=d.p_cal.mean() - k / n))
slices = pd.DataFrame(rows)
print(f"Held-out frames, per slice ({best} fitted on the pooled calibration split); gap = mean calibrated p - TP rate")
print(slices.round(3).to_string(index=False))
worst = slices.loc[slices.gap_cal.abs().idxmax()]
print(f"Largest per-slice gap after pooled calibration: {worst.cls}, {worst.band}: "
      f"mean p {worst.mean_cal:.2f} vs TP rate {worst.tp_rate:.2f} (n = {worst.n})")

# The same detections judged with a looser matching rule for cars (IoU >= 0.5 instead of 0.7)
loose = pd.DataFrame([r for fid in frames for r in match_frame(fid, frames[fid], outputs[fid],
                                                               {"Car": 0.5, "Pedestrian": 0.5})[0]])
loose = loose[(loose.status != "ignored") & (loose.cls == "Car")]
strict = det[det.cls == "Car"]
for name, d in [("IoU >= 0.7 (KITTI)", strict), ("IoU >= 0.5", loose)]:
    s9 = d[d.score >= 0.9]
    print(f"Car detections with score >= 0.9 judged at {name:18s}: TP rate {(s9.status == 'TP').mean():.2f} (n = {len(s9)})")

# Occlusion: recall of evaluable objects at a raw-score operating threshold
T_OP = 0.5
gt["detected"] = gt.score >= T_OP
gt["occlusion"] = gt.occluded.map({0: "0 fully visible", 1: "1 partly occluded", 2: "2 largely occluded"})
rec = gt.groupby(["cls", "occlusion"]).agg(n=("detected", "size"), recall=("detected", "mean"),
                                          mean_score_if_detected=("score", lambda s: s[s >= T_OP].mean()))
print(f"\nRecall at raw score >= {T_OP} by KITTI occlusion level (all 40 frames)")
print(rec.round(2).to_string())

### 6.4 Why detection-score calibration matters for AEB

In the course's AEB example (HARA in [03/02](../03_Functional_Safety_ISO26262/02_hara.ipynb)), SG-01 *avoid unintended AEB deceleration* (ASIL D) and SG-02 *avoid omission of AEB deceleration* (ASIL B) pull the decision threshold in opposite directions. The detection score enters that decision in two ways:

1. **As a threshold** ("brake only on objects with score ≥ $t$"). The rates that matter, false-activation rate per hour and miss rate per encounter, are measured at $t$ on validation data; calibration is not needed for this, but the rates must be re-measured whenever the model or the threshold changes.
2. **As a probability** in fusion (combining camera and radar evidence), in a tracker's existence probability, or in an expected-cost arbitration like Section 5.1. Then a score of 0.9 that is right only part of the time is a systematic error in every downstream computation, and the miscalibration found above propagates directly into the false-activation and omission rates that SG-01 and SG-02 constrain.

Three observations from the real detector carry over to production systems:

- **Off-the-shelf scores are not probabilities of being correct** under the evaluation rule that matters for the function; the high-score bins are where a braking decision is taken, and that is where the gap is measured above.
- **"Correct" is part of the specification.** The same detections are better or worse calibrated depending on the IoU rule; for AEB, the relevant rule is the one that makes a localisation error dangerous (lateral position error decides whether an object is in the ego path).
- **Calibration is per slice and per distribution.** A pooled calibrator can leave slices miscalibrated, and with a few hundred detections the confidence intervals are wide. KITTI is daytime, dry, German urban and suburban traffic; none of these numbers transfers to night or rain without new data (Section 5.2).

## In industry

**Work product.** Calibration results belong in the **verification and validation report of the AI component** (ISO/PAS 8800, V&V of the AI system) as evidence for a safety-related property, the quality of the component's confidence, that the AI safety requirements state per ODD slice. Where scores are consumed as probabilities (fusion, tracking, arbitration), the calibrator itself is part of the AI component: it is specified, versioned and verified with the model, and its validity is monitored in operation.

**Who produces and who reads it.** The perception ML or ML-validation engineer produces it; the sensor-fusion and AEB function developers consume it; the SOTIF / functional-safety engineer and the independent assessor review it.

**A verifiable requirement looks like this** (illustrative numbers): *"For pedestrian detections within 30 m in daylight, among detections with calibrated probability ≥ 0.8 the TP rate (KITTI-style matching, IoU ≥ 0.5) shall be at least 0.8, shown with 95 % confidence on a test set independent of training and calibration data."*

**What reviewers and assessors check**

- The matching rule (IoU threshold, class mapping, ignore regions) is stated and justified by the function, not inherited from a benchmark by default.
- Calibration, test and training data are split by drive or sequence, not by frame, and the calibration split is not the test split.
- Reliability diagrams with counts, D-ECE **and** a proper scoring rule (NLL or Brier), with confidence intervals from a cluster (per-frame or per-drive) bootstrap.
- Results per slice (class, distance band, occlusion, lighting, weather) with sample sizes; slices too thin to support a claim are named as such.
- The calibrator is frozen with the model version and re-fitted and re-verified after every retraining or sensor change.
- Operation-phase KPIs are defined, e.g. per-slice D-ECE and the TP rate of high-confidence detections on periodic labelled audits of field data, with thresholds that trigger analysis.

## Key takeaways

- Calibration compares **top-label confidence with observed accuracy**; reliability diagrams must be labelled and coloured accordingly.
- Report **ECE together with NLL and Brier**, each with a **bootstrap CI**; small ECE differences are often within the noise.
- **Temperature scaling**: one parameter, fitted by minimising validation NLL over $\log T$ in float64. $T>1$ means the model was overconfident.
- TS preserves the argmax decision; Platt (bias term) and isotonic regression can change accuracy and ranking; any calibrator changes decisions at thresholds other than 0.5.
- Fit calibrators on a validation set; report on a separate test set **and on shifted data**.
- Calibration is required for expected-cost decisions; FNR-targeted thresholds are set on validation data (or with conformal risk control) and do not need it.
- On real KITTI frames, the scores of an off-the-shelf detector are not the probability that a detection is correct under the KITTI matching rule. Recalibration fitted on separate frames narrows the gap; the matching rule and each safety-relevant slice must be checked separately.
- Detections in one frame are correlated: confidence intervals need a bootstrap over frames (or drives), not over detections.

## Self-assessment

1. What does a reliability-diagram bin *below* the diagonal mean?
2. Why is temperature scaling fitted with NLL rather than ECE?
3. Why did a float32 loss with finite-difference gradients leave $T$ at 1?
4. Which of TS, Platt and isotonic regression can change test accuracy, and why?
5. You need FNR ≤ 1 % for pedestrians. Do you need calibration to choose the threshold?
6. Why can a model calibrated on validation data be badly calibrated in fog?
7. Why must the bootstrap for detection calibration resample frames rather than individual detections?
8. A car detector is well calibrated when a TP needs IoU ≥ 0.5 but overconfident at IoU ≥ 0.7. Which result belongs in the AEB V&V report?

<details>
<summary>Answers</summary>

1. The mean confidence in that bin exceeds the accuracy: the model is overconfident there.
2. NLL is a smooth, proper scoring rule whose minimiser is the calibrated predictive distribution within the family; ECE is binned, non-differentiable, biased and depends on the binning.
3. Scaling the logits by $1/(1+10^{-8})$ does not change a float32 loss value, so the numerical gradient is exactly zero and the optimiser stops immediately at its starting point.
4. Platt (the bias $b$ moves the 0.5 boundary) and isotonic regression (piecewise-constant map creates ties and can move scores across 0.5). TS divides logits by a positive constant, which preserves their sign and order.
5. No. Any monotone score can be thresholded at the value that achieves FNR ≤ 1 % on validation data (with a statistical margin, e.g. conformal risk control). Calibration matters if the probability itself is used, e.g. in fusion or expected-cost decisions.
6. Calibration holds for the distribution it was measured on. Under shift, accuracy falls but confidence often does not fall as much, so the model becomes overconfident.
7. Detections in the same frame share lighting, scene and often the same objects, so their errors are positively correlated. Resampling detections treats them as independent and gives intervals that are too narrow; resampling frames (better: drives) keeps the dependence.
8. The one whose matching rule reflects the function's requirement. For AEB, an error matters when it changes the braking decision, e.g. a lateral box error that moves an object into or out of the ego path. The report states the rule and justifies it; if in doubt, the stricter rule or a path-based rule is used, and both results can be reported.
</details>

## References

- Guo, C., Pleiss, G., Sun, Y., Weinberger, K. Q. (2017). On calibration of modern neural networks. ICML 2017. arXiv:1706.04599
- Platt, J. (1999). Probabilistic outputs for support vector machines and comparisons to regularized likelihood methods. In *Advances in Large Margin Classifiers*, MIT Press.
- Zadrozny, B., Elkan, C. (2002). Transforming classifier scores into accurate multiclass probability estimates. KDD 2002. https://doi.org/10.1145/775047.775151
- Niculescu-Mizil, A., Caruana, R. (2005). Predicting good probabilities with supervised learning. ICML 2005. https://doi.org/10.1145/1102351.1102430
- Nixon, J. et al. (2019). Measuring calibration in deep learning. CVPR Workshops 2019. arXiv:1904.01685
- Kumar, A., Liang, P., Ma, T. (2019). Verified uncertainty calibration. NeurIPS 2019. arXiv:1909.10155
- Ovadia, Y. et al. (2019). Can you trust your model's uncertainty? Evaluating predictive uncertainty under dataset shift. NeurIPS 2019. arXiv:1906.02530
- Küppers, F. et al. (2020). Multivariate confidence calibration for object detection. CVPR Workshops 2020. arXiv:2004.13546
- Geiger, A., Lenz, P., Urtasun, R. (2012). Are we ready for autonomous driving? The KITTI vision benchmark suite. CVPR 2012. https://doi.org/10.1109/CVPR.2012.6248074
- Ren, S., He, K., Girshick, R., Sun, J. (2017). Faster R-CNN: Towards real-time object detection with region proposal networks. *IEEE TPAMI* 39(6), 1137–1149. https://doi.org/10.1109/TPAMI.2016.2577031
- Howard, A. et al. (2019). Searching for MobileNetV3. ICCV 2019. https://doi.org/10.1109/ICCV.2019.00140
- Lin, T.-Y. et al. (2014). Microsoft COCO: Common objects in context. ECCV 2014. arXiv:1405.0312
- ISO/PAS 8800:2024; ISO 21448:2022.

---
**Navigation:** ← [03 MC dropout and deep ensembles](03_mc_dropout_and_deep_ensembles.ipynb) · [Module index](README.md) · [05 Conformal prediction](05_conformal_prediction.ipynb) →

### Further practice

**Try this:** change the number of bins (5, 10, 25) and watch the ECE estimate move; train for 20 instead of 60 epochs and compare the fitted temperature; extend the decision analysis to a three-class problem with `softmax(logits, T)`.

The author's companion repository [AI-Perception-Safety-Workshop](https://github.com/milinpatel07/AI-Perception-Safety-Workshop) has further exercises.